In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os, gc, json, time, math, warnings
import numpy as np
from tqdm import tqdm
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
 
warnings.filterwarnings("ignore")
torch.set_num_threads(1)
torch.backends.cudnn.benchmark = True
 
N_GPUS = torch.cuda.device_count()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_DDP = N_GPUS > 1
 
print(f"{'='*55}")
print(f"  GPUs detected : {N_GPUS}")
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}         : {p.name}  {p.total_memory/1e9:.1f} GB")
print(f"  DataParallel  : {'ON' if USE_DDP else 'OFF'}")
print(f"{'='*55}")
 
 

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════════════════════════
 
class CFG:
    RAW_DIR  = (
        "/kaggle/input/competitions/"
        "anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/"
        "aisehack-theme-2/raw"
    )
    TEST_DIR = (
        "/kaggle/input/competitions/"
        "anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/"
        "aisehack-theme-2/test_in"
    )
 
    # ── Disk-optimised paths ──────────────────────────────────────────────
    # Store X (inputs) and Y (cpm25 target) separately instead of full
    # 26-timestep stacked array. Saves ~50 GB of disk.
    DATA_DIR  = "/kaggle/temp/data"
    TR_X      = "/kaggle/temp/data/train_X.npy"   # (N, 10, H, W, 16)
    TR_Y      = "/kaggle/temp/data/train_Y.npy"   # (N, H, W, 16)
    VA_X      = "/kaggle/temp/data/val_X.npy"
    VA_Y      = "/kaggle/temp/data/val_Y.npy"
 
    OUT_FILE  = "/kaggle/working/preds.npy"
    CKPT_DIR  = "/kaggle/working/experiments"
 
    MONTHS   = ["APRIL_16", "JULY_16", "OCT_16", "DEC_16"]
    MET_VARS = ["q2", "t2", "u10", "v10", "swdown", "pblh", "psfc", "rain"]
    EMI_VARS = ["PM25", "NH3", "SO2", "NOx", "NMVOC_e", "NMVOC_finn", "bio"]
    PMT_VAR  = ["cpm25"]
    ALL_FEAT = PMT_VAR + MET_VARS + EMI_VARS   # cpm25 first → index 0
    N_FEAT   = 16
    WIND_SET = {"u10", "v10"}
    EMIS_SET = set(EMI_VARS)
 
    T_IN    = 10
    T_OUT   = 16
    HORIZON = T_IN + T_OUT   # 26
    H, W    = 140, 124
 
    # STRIDE=2 → halves sample count vs STRIDE=1
    # Disk impact: ~22GB X + ~2GB Y = ~24GB (vs 76GB at STRIDE=1)
    STRIDE   = 2
    VAL_FRAC = 0.2
    SEED     = 42
 
    BATCH_PER_GPU = 4
    BATCH         = BATCH_PER_GPU * max(N_GPUS, 1)
 
    # Model 1 — Enhanced FNO2D
    M1_WIDTH  = 64
    M1_MODES  = 16
    M1_BLOCKS = 4
    M1_EPOCHS = 50
    M1_LR     = 2e-3
    M1_WARMUP = 5
 
    # Model 2 — ConvLSTM U-Net
    M2_BASE   = 48
    M2_HID    = 192
    M2_EPOCHS = 25
    M2_LR     = 8e-4
    M2_WARMUP = 3
 
    W1, W2    = 0.45, 0.55    # ensemble weights
    EP_ALPHA  = 4.0
    EP_CORR_W = 0.5
    LOSS_L2_W = 0.7
    LOSS_L1_W = 0.3
    PATIENCE  = 6
 
np.random.seed(CFG.SEED)
torch.manual_seed(CFG.SEED)
os.makedirs(CFG.DATA_DIR, exist_ok=True)
os.makedirs(CFG.CKPT_DIR, exist_ok=True)
 
# Disk estimate
def _disk_gb(N, shape_per_sample):
    elems = N
    for s in shape_per_sample:
        elems *= s
    return elems * 4 / 1e9
 
print(f"Effective batch : {CFG.BATCH}  ({CFG.BATCH_PER_GPU}/GPU × {N_GPUS} GPUs)")
print(f"Stride          : {CFG.STRIDE}  (reduces samples by {CFG.STRIDE}×)")
print(f"FNO2D           : width={CFG.M1_WIDTH}, modes={CFG.M1_MODES}, "
      f"blocks={CFG.M1_BLOCKS}, epochs={CFG.M1_EPOCHS}")
print(f"ConvLSTM U-Net  : base={CFG.M2_BASE}, hid={CFG.M2_HID}, "
      f"epochs={CFG.M2_EPOCHS}")
 

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# NORMALIZATION STATS  (z-score, computed from training data)
# ════════════════════════════════════════════════════════════════════════════
 
STATS_FILE = "/kaggle/working/our_stats.json"
 
def compute_stats(raw_dir, months, feat_names, out_file):
    print("Computing z-score normalization stats (sampled every 4th step)...")
    stats = {}
    for feat in tqdm(feat_names, desc="Stats"):
        chunks = []
        for month in months:
            path = os.path.join(raw_dir, month, f"{feat}.npy")
            if os.path.exists(path):
                arr = np.load(path, mmap_mode="r").astype(np.float32)
                chunks.append(arr[::4].ravel())
        if chunks:
            v = np.concatenate(chunks)
            stats[feat] = {"mean": float(v.mean()), "std": float(v.std()) + 1e-6}
        del chunks; gc.collect()
    with open(out_file, "w") as f:
        json.dump(stats, f, indent=2)
    print(f"Stats saved → {out_file}")
    return stats
 
if os.path.exists(STATS_FILE):
    print(f"Loading cached stats → {STATS_FILE}")
    with open(STATS_FILE) as f:
        STATS = json.load(f)
else:
    STATS = compute_stats(CFG.RAW_DIR, CFG.MONTHS, CFG.ALL_FEAT, STATS_FILE)
 
print(f"Stats ready: {len(STATS)} features")
 
 

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# NORMALIZE HELPERS
# ════════════════════════════════════════════════════════════════════════════
 
def normalize_feat(arr, feat):
    """Z-score normalize, with tanh for wind and clip for emissions."""
    mu, sd = STATS[feat]["mean"], STATS[feat]["std"]
    arr = (arr.astype(np.float32) - mu) / sd
    if feat in CFG.WIND_SET:
        arr = np.tanh(arr)
    if feat in CFG.EMIS_SET:
        arr = np.clip(arr, -5, 5)
    return arr
 
def denorm_cpm25(arr):
    mu, sd = STATS["cpm25"]["mean"], STATS["cpm25"]["std"]
    return arr * sd + mu
 

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# DATASET PREPARATION
# ════════════════════════════════════════════════════════════════════════════
#
# Disk layout (what we write):
#
#   train_X.npy  (N_tr, T_IN=10, H=140, W=124, N_FEAT=16)  float32
#   train_Y.npy  (N_tr, H=140,   W=124, T_OUT=16)           float32
#   val_X.npy    (N_va, T_IN,    H, W,  N_FEAT)             float32
#   val_Y.npy    (N_va, H, W,   T_OUT)                      float32
#
# Why this is better than stacked (N, 26, H, W, 16):
#   Old: N × 26 × 140 × 124 × 16 × 4B  (all features, all timesteps)
#   New: N × 10 × 140 × 124 × 16 × 4B  (X: all features, input only)
#      + N × 16 × 140 × 124 ×  1 × 4B  (Y: cpm25 only, output only)
#   Saving: we drop 16 output timesteps × 15 non-target features ≈ 50 GB
#
# ════════════════════════════════════════════════════════════════════════════
 
def build_dataset(raw_dir, months, feat_names,
                  tr_x_path, tr_y_path, va_x_path, va_y_path,
                  horizon, stride, val_frac, seed):
 
    if all(os.path.exists(p) for p in [tr_x_path, tr_y_path,
                                        va_x_path, va_y_path]):
        print("Dataset cache found — skipping preparation.")
        return
 
    print("Building X/Y split memmap dataset...")
    print(f"  Storing: X=(inputs, all {len(feat_names)} feats) "
          f"+ Y=(cpm25 target only)")
    np.random.seed(seed)
    T_IN, T_OUT = CFG.T_IN, CFG.T_OUT
    H, W = CFG.H, CFG.W
    V = len(feat_names)
    CPM25_IDX = feat_names.index("cpm25")
 
    # Pass 1: count exact samples per month (chronological split)
    month_info = {}
    for month in months:
        p0 = os.path.join(raw_dir, month, f"{feat_names[0]}.npy")
        T  = np.load(p0, mmap_mode="r").shape[0]
        n_tot = len(range(0, T - horizon + 1, stride))
        n_va  = int(n_tot * val_frac)
        n_tr  = n_tot - n_va
        month_info[month] = {"T": T, "n_tr": n_tr, "n_va": n_va}
        print(f"  {month}: T={T}, windows={n_tot} "
              f"→ train={n_tr}, val={n_va}")
 
    N_tr = sum(v["n_tr"] for v in month_info.values())
    N_va = sum(v["n_va"] for v in month_info.values())
 
    # Disk estimate
    x_gb = _disk_gb(N_tr, [T_IN, H, W, V]) + _disk_gb(N_va, [T_IN, H, W, V])
    y_gb = _disk_gb(N_tr, [H, W, T_OUT])   + _disk_gb(N_va, [H, W, T_OUT])
    print(f"\nTotal samples → train={N_tr}, val={N_va}")
    print(f"Estimated disk: X={x_gb:.1f} GB  Y={y_gb:.1f} GB  "
          f"Total={x_gb+y_gb:.1f} GB")
 
    # Allocate memmaps
    tr_X = np.memmap(tr_x_path, dtype="float32", mode="w+",
                     shape=(N_tr, T_IN, H, W, V))
    tr_Y = np.memmap(tr_y_path, dtype="float32", mode="w+",
                     shape=(N_tr, H, W, T_OUT))
    va_X = np.memmap(va_x_path, dtype="float32", mode="w+",
                     shape=(N_va, T_IN, H, W, V))
    va_Y = np.memmap(va_y_path, dtype="float32", mode="w+",
                     shape=(N_va, H, W, T_OUT))
 
    tr_ptr = va_ptr = 0
 
    for month in months:
        print(f"\nProcessing {month}...")
        info = month_info[month]
        T    = info["T"]
 
        # Load + normalize all features for this month
        feat_arrs = []
        for feat in tqdm(feat_names, desc=f"  {month} normalize"):
            path = os.path.join(raw_dir, month, f"{feat}.npy")
            if os.path.exists(path):
                feat_arrs.append(normalize_feat(np.load(path), feat))
            else:
                print(f"  [WARN] Missing {path} — filling zeros")
                feat_arrs.append(np.zeros((T, H, W), dtype=np.float32))
 
        # (T, H, W, V)
        stacked = np.stack(feat_arrs, axis=-1)
        del feat_arrs; gc.collect()
 
        starts = list(range(0, T - horizon + 1, stride))
        n_tr   = info["n_tr"]
        n_va   = info["n_va"]
 
        print(f"  Writing {n_tr} train windows...")
        for j, s in enumerate(starts[:n_tr]):
            window = stacked[s:s+horizon]               # (26, H, W, V)
            tr_X[tr_ptr + j] = window[:T_IN]            # (10, H, W, V)
            tr_Y[tr_ptr + j] = window[T_IN:, :, :, CPM25_IDX].transpose(1, 2, 0)
            # (16, H, W) → transpose → (H, W, 16)
        tr_ptr += n_tr
 
        print(f"  Writing {n_va} val windows...")
        for j, s in enumerate(starts[n_tr:n_tr+n_va]):
            window = stacked[s:s+horizon]
            va_X[va_ptr + j] = window[:T_IN]
            va_Y[va_ptr + j] = window[T_IN:, :, :, CPM25_IDX].transpose(1, 2, 0)
        va_ptr += n_va
 
        del stacked; gc.collect()
 
    tr_X.flush(); tr_Y.flush(); va_X.flush(); va_Y.flush()
    del tr_X, tr_Y, va_X, va_Y; gc.collect()
    print("\nDataset preparation complete.")
 
 
build_dataset(
    CFG.RAW_DIR, CFG.MONTHS, CFG.ALL_FEAT,
    CFG.TR_X, CFG.TR_Y, CFG.VA_X, CFG.VA_Y,
    horizon=CFG.HORIZON, stride=CFG.STRIDE,
    val_frac=CFG.VAL_FRAC, seed=CFG.SEED
)
 
# Read sample counts from file shapes
N_TR = np.memmap(CFG.TR_X, dtype="float32", mode="r").shape[0] // (
    CFG.T_IN * CFG.H * CFG.W * CFG.N_FEAT)
N_VA = np.memmap(CFG.VA_X, dtype="float32", mode="r").shape[0] // (
    CFG.T_IN * CFG.H * CFG.W * CFG.N_FEAT)
print(f"\nTrain: {N_TR} samples | Val: {N_VA} samples")
 
 

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# EPISODE MASK
# ════════════════════════════════════════════════════════════════════════════
 
def build_or_load_ep_mask(x_path, y_path, N, out_path, name):
    if os.path.exists(out_path):
        print(f"Loading {name} episode mask from cache.")
        return np.load(out_path, mmap_mode="r")
 
    print(f"Building {name} episode mask...")
    # X shape: (N, T_IN, H, W, N_FEAT) — cpm25 is feat index 0
    X = np.memmap(x_path, dtype="float32", mode="r",
                  shape=(N, CFG.T_IN, CFG.H, CFG.W, CFG.N_FEAT))
    Y = np.memmap(y_path, dtype="float32", mode="r",
                  shape=(N, CFG.H, CFG.W, CFG.T_OUT))
 
    # Use input cpm25 window to define local baseline
    inp_cpm25 = X[:, :, :, :, 0]         # (N, T_IN, H, W)
    mu  = inp_cpm25.mean(axis=1)          # (N, H, W)
    sd  = inp_cpm25.std(axis=1) + 1e-6   # (N, H, W)
 
    # Y is (N, H, W, T_OUT) — expand mu/sd to match
    mu = mu[:, :, :, np.newaxis]          # (N, H, W, 1)
    sd = sd[:, :, :, np.newaxis]          # (N, H, W, 1)
 
    mask = (Y - mu) > 1.5 * sd           # (N, H, W, T_OUT) bool
    frac = mask.mean() * 100
    print(f"  Episodic fraction: {frac:.2f}%")
 
    np.save(out_path, mask.astype(np.float32))
    del X, Y, inp_cpm25, mu, sd, mask; gc.collect()
    return np.load(out_path, mmap_mode="r")
 
ep_train = build_or_load_ep_mask(
    CFG.TR_X, CFG.TR_Y, N_TR,
    "/kaggle/working/ep_mask_train.npy", "train")
ep_val = build_or_load_ep_mask(
    CFG.VA_X, CFG.VA_Y, N_VA,
    "/kaggle/working/ep_mask_val.npy", "val")
 

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# DATASET & DATALOADERS
# ════════════════════════════════════════════════════════════════════════════
 
class PM25Dataset(Dataset):
    """
    Loads directly from X/Y split memmaps.
    No per-sample stacking needed — X already has all features pre-stacked.
    ~16× faster than per-feature loading at batch time.
    """
    def __init__(self, x_path, y_path, N, ep_mask):
        self.X  = np.memmap(x_path, dtype="float32", mode="r",
                            shape=(N, CFG.T_IN, CFG.H, CFG.W, CFG.N_FEAT))
        self.Y  = np.memmap(y_path, dtype="float32", mode="r",
                            shape=(N, CFG.H, CFG.W, CFG.T_OUT))
        self.ep = ep_mask
        self.N  = N
 
    def __len__(self): return self.N
 
    def __getitem__(self, idx):
        x  = self.X[idx].copy()    # (T_IN, H, W, N_FEAT)
        y  = self.Y[idx].copy()    # (H, W, T_OUT)
        ep = self.ep[idx].copy()   # (H, W, T_OUT)
        return (torch.from_numpy(x),
                torch.from_numpy(y),
                torch.from_numpy(ep))
 
 
kw = dict(num_workers=4, pin_memory=True,
          persistent_workers=True, prefetch_factor=3)
train_loader = DataLoader(
    PM25Dataset(CFG.TR_X, CFG.TR_Y, N_TR, ep_train),
    batch_size=CFG.BATCH, shuffle=True, **kw)
val_loader = DataLoader(
    PM25Dataset(CFG.VA_X, CFG.VA_Y, N_VA, ep_val),
    batch_size=CFG.BATCH, shuffle=False, **kw)
 
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
 
 

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# LOSS FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════
 
class RelLpLoss(nn.Module):
    def __init__(self, p=2):
        super().__init__()
        self.p = p
    def forward(self, pred, target):
        pf = pred.float().reshape(pred.shape[0], -1)
        tf = target.float().reshape(target.shape[0], -1)
        return (torch.norm(pf - tf, self.p, dim=1) /
                (torch.norm(tf, self.p, dim=1) + 1e-8)).mean()
 
 
class EpisodeAwareLoss(nn.Module):
    """
    Combined loss targeting all three evaluation metrics:
      base     = 0.7×RelL2 + 0.3×RelL1       → Global SMAPE
      e_loss   = episode-weighted MSE          → Episode SMAPE
      c_loss   = 1 - Pearson corr at episodes → Episode Correlation
    """
    def __init__(self, alpha=4.0, corr_w=0.5, l2_w=0.7, l1_w=0.3):
        super().__init__()
        self.rel_l2 = RelLpLoss(p=2)
        self.rel_l1 = RelLpLoss(p=1)
        self.alpha  = alpha
        self.corr_w = corr_w
        self.l2_w   = l2_w
        self.l1_w   = l1_w
 
    def forward(self, pred, target, ep):
        pred   = pred.float()       # (B, H, W, T_out)
        target = target.float()
        ep     = ep.float()         # (B, H, W, T_out)
 
        # Base loss — global accuracy
        base = (self.l2_w * self.rel_l2(pred, target) +
                self.l1_w * self.rel_l1(pred, target))
 
        # Episode-weighted MSE — intensity at extreme events
        w      = 1.0 + self.alpha * ep
        e_loss = (w * (pred - target).pow(2)).mean()
 
        # Correlation at episodic grid points — spatial pattern during spikes
        mask = ep.bool()
        if mask.any():
            p    = pred[mask]
            t    = target[mask]
            cov  = ((p - p.mean()) * (t - t.mean())).mean()
            corr = cov / ((p.std() + 1e-8) * (t.std() + 1e-8))
            c_loss = 1.0 - corr
        else:
            c_loss = torch.tensor(0.0, device=pred.device)
 
        return base + e_loss + self.corr_w * c_loss
 
 

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# SHARED BUILDING BLOCKS
# ════════════════════════════════════════════════════════════════════════════
 
class ChannelMLP(nn.Module):
    def __init__(self, in_ch, hid_ch=None, out_ch=None, n=2):
        super().__init__()
        hid_ch = hid_ch or in_ch
        out_ch = out_ch or in_ch
        layers = []
        for i in range(n):
            ic = in_ch  if i == 0     else hid_ch
            oc = out_ch if i == n - 1 else hid_ch
            layers.append(nn.Conv1d(ic, oc, 1))
        self.layers = nn.ModuleList(layers)
 
    def forward(self, x):
        B, C, H, W = x.shape
        x = x.view(B, C, -1)
        for i, l in enumerate(self.layers):
            x = l(x)
            if i < len(self.layers) - 1:
                x = F.gelu(x)
        return x.view(B, -1, H, W)
 
 
class SpectralConv2d(nn.Module):
    """
    Real/imag split implementation — DataParallel safe.
 
    Why this over cfloat einsum:
      - cfloat einsum with 'bixy,ioxy->boxy' requires exact 4D tensors.
        Under DataParallel, shape assumptions can break across replicas.
      - Real/imag split uses only float32 tensors throughout — no complex
        dtype ambiguity, no DataParallel broadcasting issues.
      - Captures BOTH upper and lower frequency modes (more expressive).
      - x.float() before FFT is mandatory: cuFFT does not support
        float16 on non-power-of-2 spatial dims (140×124 are not powers of 2).
    """
    def __init__(self, in_ch, out_ch, modes1, modes2):
        super().__init__()
        self.m1     = modes1
        self.m2     = modes2
        self.out_ch = out_ch
        s = 1.0 / (in_ch * out_ch)
        self.w1_re = nn.Parameter(s * torch.randn(in_ch, out_ch, modes1, modes2))
        self.w1_im = nn.Parameter(s * torch.randn(in_ch, out_ch, modes1, modes2))
        self.w2_re = nn.Parameter(s * torch.randn(in_ch, out_ch, modes1, modes2))
        self.w2_im = nn.Parameter(s * torch.randn(in_ch, out_ch, modes1, modes2))
 
    @staticmethod
    def _cmul(xr, xi, w_re, w_im):
        re = (torch.einsum("bixy,ioxy->boxy", xr, w_re) -
              torch.einsum("bixy,ioxy->boxy", xi, w_im))
        im = (torch.einsum("bixy,ioxy->boxy", xr, w_im) +
              torch.einsum("bixy,ioxy->boxy", xi, w_re))
        return re, im
 
    def forward(self, x):
        x = x.float()           # mandatory — cuFFT float32 for 140×124
        B, C, H, W = x.shape
        W2 = W // 2 + 1
 
        ft = torch.fft.rfft2(x)
        xr, xi = ft.real, ft.imag
 
        m1, m2 = self.m1, self.m2
        or_ = torch.zeros(B, self.out_ch, H, W2, device=x.device)
        oi_ = torch.zeros(B, self.out_ch, H, W2, device=x.device)
 
        # Upper modes
        re1, im1 = self._cmul(xr[:, :, :m1,  :m2], xi[:, :, :m1,  :m2],
                               self.w1_re, self.w1_im)
        or_[:, :, :m1,  :m2] = re1
        oi_[:, :, :m1,  :m2] = im1
 
        # Lower modes (captures symmetric frequency content)
        re2, im2 = self._cmul(xr[:, :, -m1:, :m2], xi[:, :, -m1:, :m2],
                               self.w2_re, self.w2_im)
        or_[:, :, -m1:, :m2] = re2
        oi_[:, :, -m1:, :m2] = im2
 
        out_ft = torch.view_as_complex(
            torch.stack([or_, oi_], dim=-1).contiguous())
        return torch.fft.irfft2(out_ft, s=(H, W)).float()
 
 
class DConv(nn.Module):
    """Double conv block with GroupNorm + GELU for U-Net."""
    def __init__(self, ic, oc):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(ic, oc, 3, padding=1, bias=False),
            nn.GroupNorm(min(8, oc), oc), nn.GELU(),
            nn.Conv2d(oc, oc, 3, padding=1, bias=False),
            nn.GroupNorm(min(8, oc), oc), nn.GELU())
 
    def forward(self, x): return self.b(x)
 
 
def pad_match(x, ref):
    """Pad x to match ref spatial dims (handles odd sizes from MaxPool2d)."""
    dh = ref.shape[2] - x.shape[2]
    dw = ref.shape[3] - x.shape[3]
    return F.pad(x, [0, dw, 0, dh]) if (dh > 0 or dw > 0) else x
 
 

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# MODEL 1: Enhanced FNO2D
# ════════════════════════════════════════════════════════════════════════════
 
class EnhFNOBlock(nn.Module):
    def __init__(self, width, modes):
        super().__init__()
        self.spectral  = SpectralConv2d(width, width, modes, modes)
        self.pointwise = nn.Conv2d(width, width, 1)
        self.mlp       = ChannelMLP(width, width // 2, width, n=2)
        self.norm1     = nn.GroupNorm(8, width)
        self.norm2     = nn.GroupNorm(8, width)
 
    def forward(self, x):
        res = x
        x   = self.norm1(self.spectral(x) + self.pointwise(x))
        x   = F.gelu(x) + res
        x   = self.norm2(x + self.mlp(x))
        return x
 
 
class EnhancedFNO2D(nn.Module):
    """
    forward(x): (B, T_in, H, W, F) → (B, H, W, T_out)
 
    Improvements over baseline:
      - width=64, modes=16, 4 blocks (vs width=48, modes=12, 3 blocks)
      - GroupNorm in every block
      - Residual connections in every block
      - ChannelMLP lifting (not single Conv2d)
      - 3-layer projection with Dropout2d
    """
    def __init__(self, time_in=10, features=16, time_out=16,
                 width=64, modes=16, n_blocks=4):
        super().__init__()
        in_ch = time_in * features + 2   # +2 for grid coords
        self.lifting = ChannelMLP(in_ch, width * 2, width, n=2)
        self.blocks  = nn.ModuleList([
            EnhFNOBlock(width, modes) for _ in range(n_blocks)])
        self.proj = nn.Sequential(
            nn.Conv2d(width, 256, 1), nn.GELU(), nn.Dropout2d(0.1),
            nn.Conv2d(256, 128, 1),  nn.GELU(),
            nn.Conv2d(128, time_out, 1))
 
    def get_grid(self, B, H, W, dev):
        gx = torch.linspace(0, 1, H, device=dev).view(1,1,H,1).expand(B,1,H,W)
        gy = torch.linspace(0, 1, W, device=dev).view(1,1,1,W).expand(B,1,H,W)
        return torch.cat([gx, gy], 1)
 
    def forward(self, x):
        B, T, H, W, F = x.shape
        # Flatten T and F into channel dim → (B, T*F, H, W)
        x = x.permute(0, 2, 3, 1, 4).reshape(B, H, W, T * F).permute(0, 3, 1, 2)
        x = torch.cat([x, self.get_grid(B, H, W, x.device)], 1)
        x = self.lifting(x)
        for blk in self.blocks:
            x = blk(x)
        return self.proj(x).permute(0, 2, 3, 1)   # (B, H, W, T_out)
 
 

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# MODEL 2: ConvLSTM U-Net
# ════════════════════════════════════════════════════════════════════════════
 
class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hid_ch, ks=3):
        super().__init__()
        self.hid_ch = hid_ch
        self.gates  = nn.Conv2d(in_ch + hid_ch, 4 * hid_ch, ks, padding=ks//2)
 
    def forward(self, x, h, c):
        i, f, o, g = self.gates(torch.cat([x, h], 1)).chunk(4, 1)
        c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
        h = torch.sigmoid(o) * torch.tanh(c)
        return h, c
 
    def init(self, B, H, W, dev):
        return (torch.zeros(B, self.hid_ch, H, W, device=dev),
                torch.zeros(B, self.hid_ch, H, W, device=dev))
 
 
class ConvLSTM(nn.Module):
    def __init__(self, in_ch, hid_ch, n=2):
        super().__init__()
        self.cells = nn.ModuleList([
            ConvLSTMCell(in_ch if i == 0 else hid_ch, hid_ch)
            for i in range(n)])
 
    def forward(self, seq):
        B, T, C, H, W = seq.shape
        states = [c.init(B, H, W, seq.device) for c in self.cells]
        for t in range(T):
            inp = seq[:, t]
            for l, cell in enumerate(self.cells):
                h, c = states[l]
                h, c = cell(inp, h, c)
                states[l] = (h, c)
                inp = h
        return states[-1][0]
 
 
class ConvLSTMUNet(nn.Module):
    """
    forward(x): (B, T_in, H, W, F) → (B, H, W, T_out)
 
    Architecture:
      U-Net encoder (shared across T timesteps)
        → ConvLSTM over bottleneck sequence (temporal context)
        → U-Net decoder with skip connections
        → head conv → output
 
    Complements FNO2D: captures local spatial structure + explicit temporal
    dynamics that FNO2D's frequency-domain approach misses.
    AMP enabled (no FFT → safe with float16 on T4).
    """
    def __init__(self, in_ch=16, base=48, hid=192, n_lstm=2, time_out=16):
        super().__init__()
        b = base
        self.enc1 = DConv(in_ch, b)
        self.enc2 = DConv(b,     b * 2)
        self.enc3 = DConv(b * 2, b * 4)
        self.pool = nn.MaxPool2d(2)
        self.lstm = ConvLSTM(b * 4, hid, n_lstm)
        self.proj = nn.Conv2d(hid, b * 4, 1)
        self.up2  = nn.ConvTranspose2d(b * 4, b * 2, 2, stride=2)
        self.dec2 = DConv(b * 4, b * 2)
        self.up1  = nn.ConvTranspose2d(b * 2, b, 2, stride=2)
        self.dec1 = DConv(b * 2, b)
        self.head = nn.Conv2d(b, time_out, 1)
 
    def encode(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        return e1, e2, e3
 
    def forward(self, x):
        # x: (B, T_in, H, W, F)
        x = x.permute(0, 1, 4, 2, 3)   # (B, T, F, H, W)
        B, T, F, H, W = x.shape
 
        bns, e1s, e2s = [], [], []
        for t in range(T):
            e1, e2, e3 = self.encode(x[:, t])
            bns.append(e3.unsqueeze(1))
            e1s.append(e1)
            e2s.append(e2)
 
        ctx = self.proj(self.lstm(torch.cat(bns, 1)))   # (B, b*4, H/4, W/4)
        e1, e2 = e1s[-1], e2s[-1]
 
        d2 = pad_match(self.up2(ctx), e2)
        d2 = self.dec2(torch.cat([d2, e2], 1))
        d1 = pad_match(self.up1(d2), e1)
        d1 = self.dec1(torch.cat([d1, e1], 1))
        return self.head(d1).permute(0, 2, 3, 1)        # (B, H, W, T_out)
 
 

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# DATAPARALLEL HELPERS
# ════════════════════════════════════════════════════════════════════════════
 
def wrap_model(model):
    model = model.to(DEVICE)
    if USE_DDP:
        model = nn.DataParallel(model, device_ids=list(range(N_GPUS)))
        print(f"  DataParallel: GPUs {list(range(N_GPUS))}")
    return model
 
def unwrap(model):
    return model.module if isinstance(model, nn.DataParallel) else model
 
 
# ════════════════════════════════════════════════════════════════════════════
# LR SCHEDULE — linear warmup + cosine decay
# ════════════════════════════════════════════════════════════════════════════
 
def make_scheduler(optimizer, epochs, warmup_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, epochs - warmup_epochs - 1)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
 
 

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# TRAINING LOOP
# ════════════════════════════════════════════════════════════════════════════
 
def train_model(model, name, save_path, epochs, lr, warmup, use_amp=False):
    criterion = EpisodeAwareLoss(
        CFG.EP_ALPHA, CFG.EP_CORR_W,
        CFG.LOSS_L2_W, CFG.LOSS_L1_W).to(DEVICE)
    opt    = torch.optim.AdamW(
        unwrap(model).parameters(), lr=lr, weight_decay=1e-4)
    sch    = make_scheduler(opt, epochs, warmup)
    scaler = torch.cuda.amp.GradScaler() if use_amp else None
    best_val = float("inf"); no_impr = 0; log = []; t0 = time.time()
 
    print(f"  AMP     : {'ON' if use_amp else 'OFF (float32 FFT safety)'}")
    print(f"  LR      : {lr} | Warmup: {warmup} ep | Total: {epochs} ep")
    print(f"  Patience: {CFG.PATIENCE}")
 
    for ep in range(1, epochs + 1):
        model.train(); tr = 0.0
        for x, y, emask in train_loader:
            x     = x.to(DEVICE, non_blocking=True)
            y     = y.to(DEVICE, non_blocking=True)
            emask = emask.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
 
            if use_amp:
                with torch.cuda.amp.autocast():
                    loss = criterion(model(x), y, emask)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(
                    unwrap(model).parameters(), 1.0)
                scaler.step(opt); scaler.update()
            else:
                loss = criterion(model(x), y, emask)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    unwrap(model).parameters(), 1.0)
                opt.step()
            tr += loss.item()
 
        sch.step(); cur_lr = opt.param_groups[0]["lr"]
 
        model.eval(); va = 0.0
        with torch.no_grad():
            for x, y, emask in val_loader:
                x     = x.to(DEVICE, non_blocking=True)
                y     = y.to(DEVICE, non_blocking=True)
                emask = emask.to(DEVICE, non_blocking=True)
                va   += criterion(model(x), y, emask).item()
 
        tr /= len(train_loader)
        va /= len(val_loader)
        elapsed = (time.time() - t0) / 60
        log.append({"ep": ep, "tr": round(tr, 5), "va": round(va, 5),
                    "lr": round(cur_lr, 8), "min": round(elapsed, 1)})
        print(f"[{name}] Ep {ep:03d}/{epochs} | lr={cur_lr:.2e} | "
              f"tr={tr:.4f}  va={va:.4f} | {elapsed:.1f}min")
 
        if va < best_val:
            best_val = va; no_impr = 0
            torch.save({
                "epoch": ep,
                "model_state_dict": unwrap(model).state_dict(),
                "val_loss": best_val}, save_path)
            print(f"  ✓ Best saved (va={va:.4f})")
        else:
            no_impr += 1
            if no_impr >= CFG.PATIENCE:
                print(f"  Early stopping at ep {ep}"); break
 
    with open(save_path.replace(".pt", "_log.json"), "w") as f:
        json.dump(log, f, indent=2)
    print(f"\n[{name}] Done. Best val={best_val:.4f}\n")
    return best_val
 
 

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# TRAIN MODEL 1 — Enhanced FNO2D
# ════════════════════════════════════════════════════════════════════════════
 
print("\n" + "=" * 60)
print("TRAINING MODEL 1 : Enhanced FNO2D (T4x2)")
print(f"  width={CFG.M1_WIDTH}, modes={CFG.M1_MODES}, "
      f"blocks={CFG.M1_BLOCKS}, epochs={CFG.M1_EPOCHS}")
print("=" * 60)
 
_m1 = EnhancedFNO2D(
    time_in=CFG.T_IN, features=CFG.N_FEAT, time_out=CFG.T_OUT,
    width=CFG.M1_WIDTH, modes=CFG.M1_MODES, n_blocks=CFG.M1_BLOCKS)
m1  = wrap_model(_m1)
p1  = sum(p.numel() for p in _m1.parameters() if p.requires_grad)
print(f"  Parameters: {p1:,}")
 
# Sanity check before committing to full training
with torch.no_grad():
    _xd = torch.randn(2, CFG.T_IN, CFG.H, CFG.W, CFG.N_FEAT).to(DEVICE)
    _od = m1(_xd)
    assert _od.shape == (2, CFG.H, CFG.W, CFG.T_OUT), \
        f"M1 shape error: {_od.shape}"
    del _xd, _od
print("  Shape check passed ✓")
 
train_model(m1, "EnhFNO2D",
            save_path=os.path.join(CFG.CKPT_DIR, "m1_best.pt"),
            epochs=CFG.M1_EPOCHS, lr=CFG.M1_LR,
            warmup=CFG.M1_WARMUP, use_amp=False)
 
del m1, _m1; gc.collect(); torch.cuda.empty_cache()
print("GPU cleared.\n")
 
 

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# TRAIN MODEL 2 — ConvLSTM U-Net
# ════════════════════════════════════════════════════════════════════════════
 
print("=" * 60)
print("TRAINING MODEL 2 : ConvLSTM U-Net (T4x2)")
print(f"  base={CFG.M2_BASE}, hid={CFG.M2_HID}, epochs={CFG.M2_EPOCHS}")
print("=" * 60)
 
_m2 = ConvLSTMUNet(in_ch=CFG.N_FEAT, base=CFG.M2_BASE,
                    hid=CFG.M2_HID, n_lstm=2, time_out=CFG.T_OUT)
m2  = wrap_model(_m2)
p2  = sum(p.numel() for p in _m2.parameters() if p.requires_grad)
print(f"  Parameters: {p2:,}")
 
# Sanity check
with torch.no_grad():
    _xd = torch.randn(2, CFG.T_IN, CFG.H, CFG.W, CFG.N_FEAT).to(DEVICE)
    _od = m2(_xd)
    assert _od.shape == (2, CFG.H, CFG.W, CFG.T_OUT), \
        f"M2 shape error: {_od.shape}"
    del _xd, _od
print("  Shape check passed ✓")
 
train_model(m2, "ConvLSTMUNet",
            save_path=os.path.join(CFG.CKPT_DIR, "m2_best.pt"),
            epochs=CFG.M2_EPOCHS, lr=CFG.M2_LR,
            warmup=CFG.M2_WARMUP, use_amp=True)
 
del m2, _m2; gc.collect(); torch.cuda.empty_cache()
print("GPU cleared.\n")
 
 

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# INFERENCE + ENSEMBLE
# ════════════════════════════════════════════════════════════════════════════
 
print("=" * 60)
print("INFERENCE & ENSEMBLE")
print("=" * 60)
 
def load_test(test_dir, feat_names):
    """Load and normalize test inputs → (N_test, T_IN, H, W, N_FEAT)."""
    arrs = []
    for feat in feat_names:
        path = os.path.join(test_dir, f"{feat}.npy")
        arr  = np.load(path).astype(np.float32)   # (N, T, H, W)
        arrs.append(normalize_feat(arr, feat)[..., np.newaxis])
    X = np.concatenate(arrs, axis=-1)              # (N, T, H, W, N_FEAT)
    print(f"Test input: {X.shape}")
    return X
 
@torch.no_grad()
def infer(model, X, bs=8):
    model.eval(); preds = []
    for s in range(0, len(X), bs):
        batch = torch.from_numpy(X[s:s+bs]).to(DEVICE)
        out   = model(batch).cpu().float().numpy()
        preds.append(out)
    return np.concatenate(preds, axis=0)   # (N, H, W, T_out)
 
X_test = load_test(CFG.TEST_DIR, CFG.ALL_FEAT)
 
# Model 1 inference
print("\nModel 1 (EnhFNO2D) inference...")
_m1 = EnhancedFNO2D(
    time_in=CFG.T_IN, features=CFG.N_FEAT, time_out=CFG.T_OUT,
    width=CFG.M1_WIDTH, modes=CFG.M1_MODES, n_blocks=CFG.M1_BLOCKS)
_m1.load_state_dict(
    torch.load(os.path.join(CFG.CKPT_DIR, "m1_best.pt"),
               map_location="cpu")["model_state_dict"])
m1 = wrap_model(_m1)
p1 = infer(m1, X_test)
del m1, _m1; gc.collect(); torch.cuda.empty_cache()
print(f"  M1 predictions: {p1.shape}")
 
# Model 2 inference
print("Model 2 (ConvLSTMUNet) inference...")
_m2 = ConvLSTMUNet(in_ch=CFG.N_FEAT, base=CFG.M2_BASE,
                    hid=CFG.M2_HID, n_lstm=2, time_out=CFG.T_OUT)
_m2.load_state_dict(
    torch.load(os.path.join(CFG.CKPT_DIR, "m2_best.pt"),
               map_location="cpu")["model_state_dict"])
m2 = wrap_model(_m2)
p2 = infer(m2, X_test)
del m2, _m2; gc.collect(); torch.cuda.empty_cache()
print(f"  M2 predictions: {p2.shape}")
 
# Weighted ensemble + denormalize + clip
print(f"\nEnsemble: {CFG.W1}×FNO2D + {CFG.W2}×ConvLSTM")
preds = CFG.W1 * p1 + CFG.W2 * p2
preds = denorm_cpm25(preds)
preds = np.clip(preds, 0, None)   # PM2.5 cannot be negative
 
# Final validation
assert preds.shape == (218, 140, 124, 16), \
    f"Shape error! Expected (218,140,124,16), got {preds.shape}"
np.save(CFG.OUT_FILE, preds.astype(np.float32))
 
print(f"""
╔══════════════════════════════════════════════════╗
║  SUBMISSION READY                                ║
╠══════════════════════════════════════════════════╣
║  File  : {CFG.OUT_FILE}
║  Shape : {preds.shape}
║  min   : {preds.min():.2f} µg/m³
║  max   : {preds.max():.2f} µg/m³
║  mean  : {preds.mean():.2f} µg/m³
╚══════════════════════════════════════════════════╝
""")
 